# Step By Step 04 - 因子预处理
--------------

在因子进入产生信号之前，我们一般都需要进行因子预处理（主要包括：标准化、去极值等）

In [ ]:
import asyncio
import datetime as dt


In [ ]:
# 首先我们准备原始数据
import numpy as np
import pyarrow as pa


## 一、标准化

In [ ]:
import pyarrow.compute as pc

rng = np.random.default_rng(42)


## 二、去极值

In [ ]:
n_dates = 6
n_assets = 12


## 三、中性化

这一步是最具有量化特色的处理。对于因子模型而言，我们一般都要对原始因子，基于**风险因子**进行中性化处理，只保留残差部分。

In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.data.processing import factor_processing


## 四、pipeline

**alpha-mind** 已经提供了一个统一的processing函数，用户可以同时将多步预处理整合在一条pipeline里面。

In [ ]:
from alphamind.data.standardize import standardize
from alphamind.data.winsorize import winsorize_normal


In [ ]:
values = np.column_stack([
    panel["factor"].to_numpy(zero_copy_only=False),
    panel["value"].to_numpy(zero_copy_only=False),
])
groups = np.unique(panel["industry"].to_pylist(), return_inverse=True)[1]


In [ ]:
risk = panel["SIZE"].to_numpy(zero_copy_only=False).reshape(-1, 1)
processed = factor_processing(
    values,
    pre_process=[winsorize_normal, standardize],
    risk_factors=risk,
    post_process=[standardize],
    groups=groups,
)


In [ ]:
assert np.isfinite(processed).all()
pa.table({"code": panel["code"], "factor": processed[:, 0], "value": processed[:, 1]})
